In [22]:
# Lancement du modele
import time

import torch
import torch.nn as nn
import torch.nn.functional as F


def get_device():
    """Renvoie le meilleur processeur disponible : cuda -> mps -> cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")      # GPU NVIDIA (Colab, cloud)
    if torch.backends.mps.is_available():
        return torch.device("mps")       # puce Apple Silicon (Mac M1-M4)
    return torch.device("cpu")           # sinon, le processeur classique


device = get_device()
torch.manual_seed(42)                    # même hasard pour tous : résultats reproductibles
print(f"PyTorch {torch.__version__} · calcul sur : {device}")
# Le chargement du corpus 
with open("fables.txt", "r", encoding="utf-8") as f:
    corpus = f.read()
    

# Ce code sert a décomposer chaque caractère du corpus en une seule fois. Par exemple, bonjour tout le monde sera décomposé en b, o, n, j, o, u, r,  , t, o, u, t,  , l, e,  , m, o, n, d, e. et les caratères seront triés en ordre alphabétique et n'apparaitront qu'une seule fois.
chars = sorted(set(corpus)) 
# print(chars)
vocab_size = len(chars)

# On crée un dictionnaire qui associe chaque caractère à un entier unique. Cela permet de convertir les caractères en entiers pour les utiliser dans le modèle. La fonction enumerate() est utilisée pour générer un index unique pour chaque caractère dans la liste chars. Le dictionnaire stoi (string to integer) est créé en utilisant une compréhension de dictionnaire, où chaque caractère c est associé à son index i.
stoi = {c: i for i, c in enumerate(chars)}

# On crée un dictionnaire qui associe chaque entier unique à son caractère correspondant. Cela permet de convertir les entiers en caractères pour interpréter les résultats du modèle. La compréhension de dictionnaire est utilisée pour inverser le dictionnaire stoi, en associant chaque index i à son caractère c.
itos = {i: c for c, i in stoi.items()} 

# Début des exercices
"""
Cette fonction encoder() prend une chaîne de caractères texte en entrée et renvoie une liste d'entiers correspondant à chaque caractère de la chaîne. Elle utilise le dictionnaire stoi pour effectuer la conversion.
"""
def encoder(texte):
    return [stoi[c] for c in texte]

"""
Cette fonction decoder() prend une liste d'entiers nombres en entrée et renvoie une chaîne de caractères correspondant à chaque entier de la liste. Elle utilise le dictionnaire itos pour effectuer la conversion.
"""
def decoder(nombres):
    return "".join(itos[i] for i in nombres)

print(f"{vocab_size} caractère distincts : {''.join(chars[1:])!r}")
print(encoder("Le loup"))
print(decoder(encoder("Le loup")))

# On convertit le corpus en une liste d'entiers en utilisant le dictionnaire stoi. Chaque caractère du corpus est remplacé par son entier correspondant. La liste d'entiers est ensuite convertie en un tenseur PyTorch pour être utilisée dans le modèle.
data = torch.tensor([stoi[c] for c in corpus])


PyTorch 2.13.0 · calcul sur : mps
81 caractère distincts : ' !"\'(),-.:;?ABCDEFGHIJLMNOPQRSTUVXYabcdefghijlmnopqrstuvxyzÀÂÇÈÉÊÔÛàâçèéêîïôùûŒœ'
[23, 40, 1, 46, 49, 55, 50]
Le loup


In [23]:
data = torch.tensor(encoder(corpus))
print(data.shape, data.dtype)
print(corpus[:24], "->", data[:24].tolist())

torch.Size([38331]) torch.int64
LA CIGALE ET LA FOURMI
L -> [23, 13, 1, 15, 21, 19, 13, 23, 17, 1, 17, 31, 1, 23, 13, 1, 18, 26, 32, 29, 24, 21, 0, 23]


In [24]:
# Créatrion du modèle

block_size = 16         # Le modele lit 16 caractères pour prédire le 17e

class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, 24)           # Chaque caractère est representé par un vecteur de 24 dimensions. Par exemple, le caractère "a" peut être représenté par le vecteur [0.1, 0.2, 0.3, ..., 0.24], tandis que le caractère "b" peut être représenté par le vecteur [0.5, 0.6, 0.7, ..., 0.24]. Ces vecteurs sont appris pendant l'entrainement du modèle.
        self.reseau = nn.Sequential(
            nn.Linear(block_size * 24, 192),        # Le vecteur de 16 caractères (16 * 24 = 384 dimensions) est transformé en un vecteur de 192 dimensions. Cela permet au modèle d'apprendre des représentations plus compactes et abstraites des séquences de caractères.
            nn.Tanh(),          # La fonction d'activation Tanh est utilisée pour introduire de la non-linéarité dans le modèle. Elle permet au modèle d'apprendre des relations complexes entre les caractères en transformant les valeurs linéaires en valeurs non linéaires comprises entre -1 et 1.
            nn.Linear(192, vocab_size),         # Le vecteur de 192 dimensions est transformé en un vecteur de vocab_size dimensions, où chaque dimension correspond à un caractère distinct dans le corpus. Cela permet au modèle de prédire la probabilité de chaque caractère possible en sortie.    
        )
    def forward(self, x):
        e = self.table(x)          # On convertit les entiers en vecteurs de 24 dimensions en utilisant la table d'embedding. Chaque entier est remplacé par son vecteur correspondant.
        return self.reseau(e.flatten(1))        # On aplatit les vecteurs de 24 dimensions pour chaque séquence de 16 caractères en un seul vecteur de 384 dimensions (16 * 24 = 384). Cela permet au modèle de traiter la séquence entière comme une seule entrée. Ensuite, on passe ce vecteur à travers le réseau de neurones pour obtenir les scores de prédiction pour chaque caractère possible en sortie.
    
    
model = MiniLM().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} nombres à apprendre dans le modèle")

91,497 nombres à apprendre dans le modèle


In [25]:
x_test = torch.randint(0, vocab_size, (4, block_size), device=device)
scores = model(x_test)

assert isinstance(scores, torch.Tensor), "le forward doit renvoyer un tenseur"      # On vérifie que la sortie du modèle est bien un tenseur PyTorch. Si ce n'est pas le cas, une assertion est levée avec le message "le forward doit renvoyer un tenseur".
assert scores.shape == (4, vocab_size), (
    f"shape attendue (4, {vocab_size}), obtenue {tuple(scores.shape)}"
)       # On vérifie que la forme du tenseur de sortie est correcte. La forme attendue est (4, vocab_size), où 4 correspond au nombre de séquences d'entrée et vocab_size correspond au nombre de caractères distincts dans le corpus. Si la forme n'est pas correcte, une assertion est levée avec un message indiquant la forme attendue et la forme obtenue.

print("Forward OK : 4 contextes -> 4 lignes de", vocab_size, "scores")


Forward OK : 4 contextes -> 4 lignes de 81 scores


In [26]:
# Les fonctions de génération et d'entrainement du modèle.

def generer(prompt="\n", longueur=300, temperature=1.0):
    model.eval()        # On met le modèle en mode évaluation pour désactiver certaines fonctionnalités spécifiques à  l'entrainement, comme le dropout. Cela permet d'obtenir des résultats plus stables lors de la génération de texte.
    
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]         # On crée un contexte initial en utilisant le caractère de nouvelle ligne ("\n") répété block_size fois, suivi des entiers correspondant aux caractères du prompt. Ensuite, on prend les block_size derniers éléments de cette liste pour former le contexte initial. Cela permet au modèle de commencer la génération à partir d'un état initial cohérent.
    sortie = []
    with torch.no_grad():        # On désactive le calcul des gradients pour économiser de la mémoire et accélérer les calculs, car nous n'avons pas besoin de mettre à jour les poids du modèle lors de la génération de texte.
        for _ in range(longueur):
            x = torch.tensor([ctx], device=device)       # On convertit le contexte en un tenseur PyTorch et on le place sur le même appareil que le modèle (CPU ou GPU). Cela permet au modèle de traiter le contexte comme une entrée.    
            
            probas = F.softmax(model(x) / temperature, dim=-1)        # On applique la fonction softmax aux scores de sortie du modèle pour obtenir des probabilités normalisées pour chaque caractère possible. La température est utilisée pour contrôler la "créativité" de la génération : une température plus élevée rend les probabilités plus uniformes, tandis qu'une température plus basse rend les probabilités plus concentrées sur les caractères les plus probables.
            
            i = torch.multinomial(probas, num_samples=1).item()         # On échantillonne un caractère à partir des probabilités obtenues en utilisant la fonction torch.multinomial(). Cela permet de choisir un caractère de manière aléatoire en fonction des probabilités prédites par le modèle.
            sortie.append(itos[i])
            ctx = ctx[1:] + [i]         # On met à jour le contexte en supprimant le premier caractère et en ajoutant le caractère échantillonné à la fin. Cela permet au modèle de générer du texte de manière séquentielle, en utilisant les caractères précédemment générés comme contexte pour prédire le caractère suivant.
    return prompt + "".join(sortie)


In [27]:
essai = generer("Le loup", longueur=50)
assert essai.startswith("Le loup"), "La sortie doit commencer par le prompt"        # On vérifie que la sortie générée par le modèle commence bien par le prompt fourni. Si ce n'est pas le cas, une assertion est levée avec le message "La sortie doit commencer par le prompt". Cela permet de s'assurer que le modèle respecte correctement le contexte initial lors de la génération de texte.

# ? Explique moi le role de la focntion assert dans le code ci-dessus. 
# La fonction assert est utilisée pour vérifier une condition spécifique dans le code. Si la condition est vraie, le programme continue son exécution normalement. Cependant, si la condition est fausse, une exception AssertionError est levée, et le message fourni (dans ce cas, "La sortie doit commencer par le prompt") est affiché. Cela permet de détecter rapidement les erreurs ou les comportements inattendus dans le code, en s'assurant que certaines hypothèses sont respectées. Dans ce contexte, l'assertion vérifie que la sortie générée par le modèle commence bien par le prompt fourni, garantissant ainsi que le modèle fonctionne comme prévu.

assert len(essai) == len("Le loup") + 50, "La sortie doit contenir ‘longueur‘ caractères de plus"       # On vérifie que la longueur de la sortie générée par le modèle est égale à la longueur du prompt plus la valeur spécifiée pour l'argument longueur. Si ce n'est pas le cas, une assertion est levée avec le message "La sortie doit contenir ‘longueur‘ caractères de plus". Cela permet de s'assurer que le modèle génère le nombre correct de caractères supplémentaires après le prompt initial.

assert set(essai) <= set(chars), "tous les caractères générés doivent venir du vocabulaire" 

print("Echantillonnage OK :", essai[:40].replace("\n", " ") + "…" )

Echantillonnage OK : Le loupâYyÔ?;NulÀ?H)ï G,ùgàqDoŒouÀ(à,âi'…


In [28]:
print(generer("Le loup", longueur=300))

Le loupŒâXNÂÛhohD!g)IBÇDrXœmpxPêNxmM"j
ÔièM.ôSh:nx çn.vQYVÂà,È?N:r-:Jc.B;Ê;ÈUîJ?ÈvôHÛdqîDôdAXfï)eo?èÊX(TMJpEfyOtJâï?éxUÂXéJïâ)vQqQç)Jzj"hsLMqhnzaJâûïPàÀn:JUQràùÀVAo o"ùâûç,ÈÊ?Sê!SâèjzqN.gUî
NOÊjC)Ê:SOv:ÀAHBgé(iY"d:È,ŒreÀzï-YxçFEçjéisXÔœf-SuO:"yipÀiïÔX
çJiïbŒJâ:ÛôÊbFMà(C!?QUfvéaÔooRmNêNÀCœ Osl!èUbRâ:ÛàYhvûç


In [29]:
def fabriquer_batch(taille=64):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))            # On génère un tenseur ix contenant des indices aléatoires compris entre 0 et len(data) - block_size - 1. Ces indices sont utilisés pour sélectionner des séquences de caractères dans le tenseur data. La taille du batch est déterminée par l'argument taille, qui spécifie le nombre de séquences à générer.
    x = torch.stack([data[i : i + block_size] for i in ix]).to(device)       # On crée un tenseur x contenant des séquences de caractères de longueur block_size. Chaque séquence est extraite du tenseur data en utilisant les indices aléatoires générés dans ix. Le tenseur x est ensuite déplacé vers l'appareil (CPU ou GPU) utilisé par le modèle.
    y = data[ix + block_size].to(device)        # On crée un tenseur y contenant les caractères cibles correspondants à chaque séquence de caractères dans x. Chaque caractère cible est extrait du tenseur data en utilisant les indices aléatoires générés dans ix, mais décalés de block_size pour obtenir le caractère suivant dans la séquence. Le tenseur y est ensuite déplacé vers l'appareil (CPU ou GPU) utilisé par le modèle.
    return x, y

x, y = fabriquer_batch()        # On fabrique un batch de données d'entraînement en appelant la fonction fabriquer_batch(). Cette fonction génère un ensemble de séquences de caractères (x) et leurs caractères cibles correspondants (y) à partir du corpus de texte. Le batch est utilisé pour entraîner le modèle en lui fournissant des exemples d'entrée et de sortie.

print("contextes :", x.shape, "| cibles :", y.shape)        # On affiche la forme des tenseurs x et y pour vérifier que les dimensions sont correctes. Le tenseur x contient les séquences de caractères d'entrée (contextes), tandis que le tenseur y contient les caractères cibles correspondants. Cela permet de s'assurer que les données sont correctement préparées pour l'entraînement du modèle.

print(repr(decoder(x[0].tolist())), "->", repr(itos[y[0].item()]))      # On décode la première séquence de caractères d'entrée (x[0]) en utilisant la fonction decoder() pour obtenir la représentation en chaîne de caractères. Ensuite, on récupère le caractère cible correspondant (y[0]) et on utilise le dictionnaire itos pour obtenir le caractère associé à cet entier. Enfin, on affiche la séquence d'entrée décodée et le caractère cible correspondant pour vérifier que les données sont correctement préparées pour l'entraînement du modèle.

contextes : torch.Size([64, 16]) | cibles : torch.Size([64])
'teint le faîte ;' -> '\n'


In [30]:

# Boucle pour l'entrainement du modèle
optimiseur = torch.optim.AdamW(model.parameters(), lr=1e-3)

debut = time.time()
for step in range(8001):
    x, y = fabriquer_batch()
    scores = model(x)       # Prédiction des scores pour chaque caractère possible en sortie, en utilisant les séquences de caractères d'entrée (x) comme contexte. Le modèle calcule les scores pour chaque caractère dans le vocabulaire, ce qui permet de déterminer la probabilité de chaque caractère étant donné le contexte fourni.
    
    loss = F.cross_entropy(scores, y)       # Mesure de l'erreur du modèle en comparant les scores prédits avec les caractères cibles réels (y). La fonction F.cross_entropy() calcule la perte d'entropie croisée, qui est une mesure de la différence entre les distributions de probabilité prédites et les distributions de probabilité réelles. Une perte plus faible indique que le modèle fait de meilleures prédictions.
    
    optimiseur.zero_grad()      # Remise du gradient à zéro pour éviter l'accumulation des gradients d'itérations précédentes. Cela permet de s'assurer que les gradients calculés lors de la rétropropagation ne sont pas influencés par les gradients des itérations précédentes, ce qui pourrait fausser la mise à jour des poids du modèle.
    
    loss.backward()             # Calcule des corrections à apporter aux poids du modèle en fonction de l'erreur mesurée par la perte. La méthode backward() effectue la rétropropagation, qui calcule les gradients des poids du modèle par rapport à la perte. Ces gradients sont ensuite utilisés pour mettre à jour les poids du modèle afin d'améliorer ses performances lors de l'entraînement.
    
    optimiseur.step()           # Correction des poids du modèle en fonction des gradients calculés lors de la rétropropagation. La méthode step() applique les mises à jour des poids en utilisant l'optimiseur (dans ce cas, AdamW) pour ajuster les poids du modèle afin de réduire la perte et améliorer les performances du modèle lors de l'entraînement.
    
    if step % 1000 == 0:
        print(f"étape {step:5d} | loss {loss.item():.2f}")

print(f"Entrainement terminé en {time.time() - debut:.0f} s")

étape     0 | loss 4.42
étape  1000 | loss 2.07
étape  2000 | loss 1.96
étape  3000 | loss 1.42
étape  4000 | loss 1.32
étape  5000 | loss 1.15
étape  6000 | loss 1.15
étape  7000 | loss 0.95
étape  8000 | loss 0.75
Entrainement terminé en 30 s


In [31]:
with torch.no_grad():
    pertes = []
    for _ in range(20):
        x, y = fabriquer_batch()
        pertes.append(F.cross_entropy(model(x), y).item())
    loss_moyenne = sum(pertes) / len(pertes)
print(f"loss moyenne après entrainement : {loss_moyenne:.2f}")
assert loss_moyenne < 2.0, (
    f"loss {loss_moyenne:.2f} trop haut: la boucle n'a pas appris (attendu < 2.0)"
)
print("Entrainement OK : le modele a appris quelque chose.")
        

loss moyenne après entrainement : 0.91
Entrainement OK : le modele a appris quelque chose.


In [32]:
print(generer("Le loup", longueur=400, temperature=0.5))

Le loupré ! vois tout les plut que l'écaler
À quelque aus bour le veut che aux démends etsantants d'un faurait par tacet de meun,
Sant bouché, pros, da sous, le sons pas s pouts coun mes reur tet nous marson dressi seurelle tentaux destemplesse.
Ll Et En se la deur de ma vien,
S'antait ces rendins, quelle roille ron de tre dire la feure,
Ni ces ant que de ne forte : sa fon des-vous ;
Qu'ellot dine lui fa


In [33]:
for _ in [0.5, 1.0, 1.3]:
    print(f"----- temperature = {_} -----")
    print(generer("La cigale", longueur=200, temperature=_))
    print()    

----- temperature = 0.5 -----
La cigale ropon qu'il faut d're vous de marges.
Je souffoir plut que à fon cale avecherce dusent,
Lu se son frac, de feurles, par tour le faire,
Si ventre lu doufant pas rois.
Le toupre et la fêtéfoil ; et ven

----- temperature = 1.0 -----
La cigale rouc : estust iepss-
Ne toupllos, hondre m'harbie ;
On re la mien qui vait felore vait voyt.
Le fit déjà pas sognes,
Larde lui fétend,
Trut qu'ulet la plaiglaîfe ;
L'hevreT :
L'Et HER CeRT fUn à Gièp

----- temperature = 1.3 -----
La cigalel voicher que uvaunntobre :
Se det-moi,
Pror un vous tuês quellau té landêt : chattemuc qu'à j'evaitser pagtop dom bien d'ochequin drnil : qui ce te celle pon flavittuinnrerouse.
C'ériet la mit : ce,




# Chapitre 1 
## Exercices

### 1 - Le forward, à la main

In [34]:
def forward_manuel(x):
    e = model.table(x)
    return model.reseau(e.flatten(1))
    
    
x_test = torch.randint(0, vocab_size, (4, block_size), device=device)
scores = forward_manuel(x_test)
assert isinstance(scores, torch.Tensor), "forward_manuel doit renvoyer un tenseur"
assert scores.shape == (4, vocab_size), (
    f"shape attendue (4, {vocab_size}), obtenue {tuple(scores.shape)}"
)
assert torch.allclose(scores, model(x_test)), (
    "les scores doivent être identiques à ceux de model(x_test)"
)
print("Forward OK : 4 contextes -> 4 lignes de", vocab_size, "scores, identiques au modèle")

Forward OK : 4 contextes -> 4 lignes de 81 scores, identiques au modèle


### 2 Échantillonner moi-même

In [35]:
def ma_generation(prompt="\n", longueur=300, temperature=1.0):
    model.eval()
    ctx = ([stoi["\n"]] * block_size + encoder(prompt))[-block_size:]
    sortie = []
    with torch.no_grad():
        for _ in range(longueur):
            x = torch.tensor([ctx], device=device)
            scores_manuel = model(x)
            probs = F.softmax(scores_manuel / temperature, dim=-1)
            num_hasard = torch.multinomial(probs, num_samples=1).item()
            sortie.append(itos[num_hasard])
            ctx = ctx[1:] + [num_hasard]
        model.train()
        return prompt + "".join(sortie)
    
# Validation : échantillonnage. Bonne longueur, caractères du vocabulaire uniquement.
essai = ma_generation("Le loup", longueur=50)
assert essai.startswith("Le loup"), "la sortie doit commencer par le prompt"
assert len(essai) == len("Le loup") + 50, "la sortie doit contenir `longueur` caractères de plus"
assert set(essai) <= set(chars), "tous les caractères générés doivent venir du vocabulaire"
print("Échantillonnage OK :", essai[:40].replace("\n", " ") + "…")

Échantillonnage OK : Le louppla sois jeura cempère. L'ompout …


### 3 - Le refrain d'entrainement

In [36]:
model2 = MiniLM().to(device)
optimiseur2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)

for step in range(8001):
    x, y = fabriquer_batch()
    scores = model2(x)
    loss = F.cross_entropy(scores, y)
    optimiseur2.zero_grad()
    loss.backward()
    optimiseur2.step()
    if step % 1000 == 0:
        print(f"étape {step:5d} | loss {loss.item():.2f}")
        

# Validation : entraînement. La loss doit être bien plus basse qu'au départ (~4.4).
with torch.no_grad():
    pertes = []
    for _ in range(20):
        x, y = fabriquer_batch()
        pertes.append(F.cross_entropy(model2(x), y).item())
    loss_moyenne = sum(pertes) / len(pertes)
print(f"loss moyenne de model2 après entraînement : {loss_moyenne:.2f}")
assert loss_moyenne < 2.0, (
    f"loss {loss_moyenne:.2f} trop haute : la boucle n'a pas appris (attendu < 2.0)"
)
print("Entraînement OK : ton refrain fonctionne.")

étape     0 | loss 4.42
étape  1000 | loss 2.02
étape  2000 | loss 2.04
étape  3000 | loss 1.52
étape  4000 | loss 1.49
étape  5000 | loss 1.31
étape  6000 | loss 1.17
étape  7000 | loss 0.70
étape  8000 | loss 0.82
loss moyenne de model2 après entraînement : 0.89
Entraînement OK : ton refrain fonctionne.


# Chapitre 2 - Les nombres qui apprennent
## Théorie et pratique

In [37]:
# Après le code dans le chapitre 1, je vais juste sauvegarder les données d'entrainement du modèle afin de continuer
torch.save(model.state_dict(), "miniLm.pt")

etat = torch.load("miniLm.pt")
total = 0
for nom, t in etat.items():
    print(f"{nom:16s} shape {str(tuple(t.shape)):12s} {t.numel():6d} nombres")
    total += t.numel()
    
print(f"{'':16s} {'':18s} -------")
print(f"{'total':16s} {':18s'} {total:6d} nombres")
assert total == 91_497, "le fichier de MiniLM doit contenir 91 497 nombres"
assert len(etat) == 5, "cinq paquets : trois matrices et deux vecteurs"

table.weight     shape (81, 24)       1944 nombres
reseau.0.weight  shape (192, 384)    73728 nombres
reseau.0.bias    shape (192,)          192 nombres
reseau.2.weight  shape (81, 192)     15552 nombres
reseau.2.bias    shape (81,)            81 nombres
                                    -------
total            :18s  91497 nombres


## 2 - Le vecteur
On utilise un exemple terre a terre ici pour l'expliquer. Ayélé qui fait du. jus frais d'ananas. Pour sa recette, elle prend 3 ananas pour 2 poignées de gingembre. L'ananas coute 250xof a l'unité et le gingembre 100xof

En PyTorch, la liste devient un tenseur

In [38]:
import torch
doses = torch.tensor([3.0, 2.0])
prix = torch.tensor([250.0, 100.0])
print(doses.shape)

torch.Size([2])


In [39]:
lundi = torch.tensor([3.0, 2.0])
mardi = torch.tensor([1.0, 4.0])

print(lundi + mardi)
print(lundi * 10)

tensor([4., 6.])
tensor([30., 20.])


## 3 - Le produit scalaire

En notation: $\mathbf{d} \cdot \mathbf{p} = \sum_i d_i\, p_i$, « multiplie
$d_i$ par $p_i$ pour chaque position $i$, et additionne tout »

In [40]:
print(torch.dot(doses, prix))
print(doses * prix)
print((doses * prix).sum())

tensor(950.)
tensor([750., 200.])
tensor(950.)


In [41]:
droite = torch.tensor([1.0, 0.0])
haut = torch.tensor([0.0, 1.0])
gauche = torch.tensor([-1.0, 0.0])

print(torch.dot(droite, droite))
print(torch.dot(droite, haut))
print(torch.dot(droite, gauche))

tensor(1.)
tensor(0.)
tensor(-1.)


## 4 - La matrice 

In [43]:
D = torch.tensor([[3.0, 2.0], [1.0, 4.0]])                    # Doses: 2 recettes, 2 ingrédients
P = torch.tensor([[250.0, 400.0], [100.0, 150.0]])            # Prix: 2 prix, 2 fournisseurs 

def multiplication_matrice_a_la_main(A, B):
    m1, n1 = A.shape            # A: m lignes, n colonnes
    m2, n2 = B.shape            # B: m lignes, n colonnes
    assert n1 == m2, "les dimensions du milieu doivent se rencontrer"
    C = torch.zeros(m1, n2)
    for i in range(m1):                     # pour chaque ligne de A…
        for j in range(n2):                     # …et chaque colonne de B…
            for k in range(n1):                 # …un produit scalaire complet
                C[i, j] += A[i, k] * B[k, j]
    return C

print(multiplication_matrice_a_la_main(D, P))
print(D @ P)

tensor([[ 950., 1500.],
        [ 650., 1000.]])
tensor([[ 950., 1500.],
        [ 650., 1000.]])


La multiplication de deux matrices suit certaines règles:
D(m, n) et P(n, p). Les n doivent etre les mêmes et ainsi on aura C = DP => C(m, p)

In [45]:
prix_bizarre = torch.tensor([[250.0, 400.0, 350.0]])
print(prix_bizarre.shape)

try:
    D @ prix_bizarre
except RuntimeError as e:
    message = str(e)
    print("RuntimeError:", message)
    
assert "cannot be multiplied" in message, "PyTorch doit refuser ce produit"
print("Erreur provoquée et comprise: les dimensions du milieu , 2 et 1 ne se rencontrent pas")    


torch.Size([1, 3])
RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x2 and 1x3)
Erreur provoquée et comprise: les dimensions du milieu , 2 et 1 ne se rencontrent pas


In [53]:
M = torch.tensor([[1.5, 0.5], 
                  [0.4, 1.3]])
print(M.shape)
droite = torch.tensor([1.0, 0.0])
haut = torch.tensor([0.0, 1.0])
print(droite.shape)
print(M @ droite)
print(M @ haut)

assert torch.allclose(M @ droite, M[:, 0]), "[1, 0] atterrit sur la 1er colonne"
assert torch.allclose(M @ haut, M[:, 1]), "[0, 1] atterrit sur la 2er colonne"

torch.Size([2, 2])
torch.Size([2])
tensor([1.5000, 0.4000])
tensor([0.5000, 1.3000])


### 6 - MiniLM shape par shape

.T -> Transposition d'une matrice

In [58]:
a = torch.randn(64, 384, device=device)
W1 = etat["reseau.0.weight"]        # (192, 384)
b1 = etat["reseau.0.bias"]         # (192,)

z = a @ W1.T + b1
print(z.shape)
assert z.shape ==(64, 192), "règle d'or : (64, 384) @ (384, 192) donne (64, 192)"

W2 = etat["reseau.2.weight"]         # (81, 192)  
b2 = etat["reseau.2.bias"]          # (81,)

scores = torch.tanh(z) @ W2.T + b2
print(scores.shape)
assert scores.shape == (64, 81), "81 scores, un par caractère possible pour la suite"

couche_1 = 64 * 192 * 384
couche_2 = 64 * 81 * 192
assert couche_1 == 4_718_592 and couche_2 == 995_328
print(f"multiplication par lot : {couche_1 + couche_2:,} (environ 5,7 millions)")

torch.Size([64, 192])
torch.Size([64, 81])
multiplication par lot : 5,713,920 (environ 5,7 millions)


In [59]:
colonne = torch.tensor([[1.0], [2.0], [3.0]])
lignes = torch.tensor([[10.0, 20.0, 30.0]])
print((colonne + lignes).shape)

assert(colonne + lignes).shape == (3, 3), (
    "Le bug qui ne crie pas : chaque dimension de taille 1 est étirée fauce 3 d'en face"
)
print("Tu attendais 3 nombres, tu obtiens une matrice 3 x 3, et rien plante.")
print("L'antidote : le reflexe shapa, après chaque opération")

torch.Size([3, 3])
Tu attendais 3 nombres, tu obtiens une matrice 3 x 3, et rien plante.
L'antidote : le reflexe shapa, après chaque opération
